## Processament del llenguatge natural. Exemple 01

Anem a veure un exemple senzill de processament de llenguatge natural. Utilitzarem Python i la llibreria `nltk` per analitzar un text i extreure informació útil.

### La llibreria `nltk`

La llibreria `nltk` (Natural Language Toolkit) és una de les eines més populars per al processament del llenguatge natural en Python. Permet realitzar tasques com tokenització, etiquetatge gramatical, anàlisi sintàctica, i molt més.


In [ ]:
# Si no tenim instal·lada la llibreria `nltk`, podem instal·lar-la amb el següent comandament:

!pip install nltk

In [13]:
# Ara importem la llibreria i descarreguem alguns recursos
# Esta part no cal executar-la cada vegada, només la primera vegada que utilitzem `nltk` 
# per descarregar els models necessaris.

import nltk
import string
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.util import ngrams
from nltk.probability import FreqDist
from nltk.collocations import BigramCollocationFinder
from collections import defaultdict

# Descarreguem els models necessaris (només ho farà una vegada, després detecta que ja els tenim baixats)
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package punkt_tab to /home/fidel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/fidel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/fidel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/fidel/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/fidel/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/fidel/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

### Corpus per fer proves

Anem a baixar El Quijote en format txt per provar coses en un text amb certa grandària i veure si les operacions se fan de forma ràpida o no.

In [ ]:
import requests

url = "https://babel.upm.es/~angel/teaching/pps/quijote.txt"
resp = requests.get(url, verify=False)  # ← Ignora SSL
resp.raise_for_status()

texto = resp.text
print(texto[:500])  # Primeros 500 chars
print("\nLongitud:", len(texto))


El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho
libro a tres maravedís y medio; el cual tiene ochenta y tres pliegos, que
al dicho precio monta el dicho libro docientos y noventa maravedís y medio,
en q

Longitud: 2097953


### Expressions regulars per a la tokenització

Les expressions regulars són patrons que ens permeten identificar i manipular cadenes de text. 

En el context de la tokenització, les expressions regulars poden ser molt útils per a definir com
volem dividir el text en tokens. També per filtrar o identificar certs tipus de tokens, com ara paraules, números, o puntuació.

Per exemple, podem utilitzar les següents expressions regulars per a tokenitzar un text en paraules:
- `\w+`: Esta expressió regular coincideix amb qualsevol seqüència de caràcters alfanumèrics (paraules). 
- `\d+`: Esta expressió regular coincideix amb qualsevol seqüència de dígits (números). 
- `[^\w\s]`: Esta expressió regular coincideix amb qualsevol caràcter que no siga una paraula o un espai (puntuació).

In [17]:
# Utilització d'expressions regulars prèviament a la tokenització
# Anem per exemple a fer que només considere paraules alfanumèriques (sense signes de puntuació
# ni caràcters especials). Això normalment ajuda a millorar la qualitat del tractament del text.

from nltk.tokenize import RegexpTokenizer
tokenizer = RegexpTokenizer(r'\w+')



### Tokenització

Exemple de tokenització de frases i paraules. Com ja sabeu, la tokenització és el procés
de dividir un text en unitats més xicotetes, com ara frases o paraules.


In [26]:

# Tokenització per oracions
sentences = sent_tokenize(texto, language="spanish") # si no especiquem expressió regular, per defecte utilitza
# el model de tokenització per oracions que té integrat, que és bastant bo i funciona bé en la majoria de casos.
# mostrem les primeres 5 oracions
# cada oració en una línia
for sentence in sentences[:5]:
    print(sentence)
    # un separador, per veure on acaba cada oració
    print("-" * 50)

# Tokenització per paraules
tokens = word_tokenize(texto.lower(), language="spanish") 
print(f"\nTokens:", tokens[:20])  # Primeres 20 paraules

# Utilitzant l'expressió regular per tokenitzar només paraules alfanumèriques
tokens_regex = tokenizer.tokenize(texto.lower()) 
print(f"\nTokens amb regex:", tokens_regex[:20]) # Primeres 20 paraules

El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho
libro a tres maravedís y medio; el cual tiene ochenta y tres pliegos, que
al dicho precio monta el dicho libro docientos y noventa maravedís y medio,
en que se ha de vender en papel; y dieron licencia para que a este precio
se pueda vender, y mandaron que esta tasa se ponga al principio del dicho
libro, y no se pueda vender sin ella.
--------------------------------------------------
Y, para que dello conste, di la
presente en Valladolid, a veinte días del mes de deciembre de mil y
seiscientos y cuatro años.
--------------------------------------------------
Juan Gallo de Andrada.
--------------------------------------------------
TESTIMONIO DE L

### Etiquetar

L'etiquetatge gramatical és el procés d'assignar una etiqueta a cada paraula d'un text que indica la seua categoria gramatical (substantiu, verb, adjectiu, etc.). Això és molt útil per a l'anàlisi sintàctica i semàntica del text, ja que ens permet comprendre la funció de cada paraula en la frase i com es relaciona amb les altres paraules. Per exemple, en la frase "El gat menja el peix", l'etiquetatge gramatical ens permet identificar que "gat" és un substantiu, "menja" és un verb, i "peix" és un substantiu. Això ens ajuda a comprendre que el gat és el subjecte que realitza l'acció de menjar, i el peix és l'objecte que rep l'acció.

Podeu consultar les principals etiquetes gramaticals que utilitza NLTK en este enllaç:

https://www.geeksforgeeks.org/python/part-speech-tagging-stop-words-using-nltk-python/


In [44]:
# Anem a etiquetar les primeres 50 paraules del text amb les seues categories gramaticals
# Gastem tokens_regex perquè ja hem eliminat els signes de puntuació i altres caràcters especials.
tokens_to_tag = tokens_regex[:50] # els primers 50 tokens
tags = pos_tag(tokens_to_tag) # creació de les etiquetes gramaticals
for paraula, tag in tags:
    print(f"{paraula}: {tag}")

# Podreu comprovar que l'etiquetador de NLTK funciona prou mal en castellà. Després provarem altres opcions.


el: NN
ingenioso: NN
hidalgo: NN
don: JJ
quijote: NN
de: IN
la: FW
mancha: FW
tasa: NN
yo: NN
juan: JJ
gallo: NN
de: IN
andrada: FW
escribano: FW
de: FW
cámara: FW
del: FW
rey: FW
nuestro: JJ
señor: NN
de: IN
los: FW
que: JJ
residen: NN
en: NN
su: NN
consejo: NN
certifico: NN
y: NN
doy: NN
fe: NN
que: NN
habiendo: NN
visto: NN
por: NN
los: NN
señores: VBZ
dél: JJ
un: JJ
libro: NN
intitulado: NN
el: NN
ingenioso: JJ
hidalgo: NN
de: IN
la: FW
mancha: FW
compuesto: NN
por: NN


### Stemming. Lemmització. Stop words.

El ***stemming*** és el procés de reduir les paraules a la seua arrel
La ***lemmatització*** és el procés de reduir les paraules a la seua forma canònica (lemma).
Anem a veure un exemple en el que el resultat del stemming i el resultat de la lemmatització són diferents:
En teoria, la paraula "corriendo" en castellà se reudiria a:

- ***Stemming***: "corr"
- ***Lemmatització***: "correr"

Els ***stop words*** són paraules que es consideren poc informatives i que normalment s'eliminen del text abans de l'anàlisi. En castellà, algunes stop words comunes són "el", "la", "de", "y", etc.

In [45]:
# Anem a carregar els stop words en castellà.

stop_words = set(stopwords.words('spanish'))

# Anem a filtar els tokens que ja hem generat, però ara sense stop words
filtered_tokens = [w for w in tokens_regex if w not in stop_words]

# Tornem a mostrar els 50 primers tokens, ara sense les paraules de parada (stop words)
for token in filtered_tokens[:50]:
    print(token)


ingenioso
hidalgo
don
quijote
mancha
tasa
juan
gallo
andrada
escribano
cámara
rey
señor
residen
consejo
certifico
doy
fe
visto
señores
dél
libro
intitulado
ingenioso
hidalgo
mancha
compuesto
miguel
cervantes
saavedra
tasaron
cada
pliego
dicho
libro
tres
maravedís
medio
ochenta
tres
pliegos
dicho
precio
monta
dicho
libro
docientos
noventa
maravedís
medio


In [46]:
# Ara anem a fer stemming

# Stemming (Porter). És un dels algoritmes de stemming més coneguts i utilitzats.
porter = PorterStemmer()
stemmed_tokens = [porter.stem(w) for w in filtered_tokens]

# Mostrem els 50 primers tokens després de fer stemming
for token in stemmed_tokens[:50]:
    print(token)


ingenioso
hidalgo
don
quijot
mancha
tasa
juan
gallo
andrada
escribano
cámara
rey
señor
residen
consejo
certifico
doy
fe
visto
señor
dél
libro
intitulado
ingenioso
hidalgo
mancha
compuesto
miguel
cervant
saavedra
tasaron
cada
pliego
dicho
libro
tre
maravedí
medio
ochenta
tre
pliego
dicho
precio
monta
dicho
libro
dociento
noventa
maravedí
medio


In [47]:

# Porter funciona bé en anglès, en castellà no tant.
# Anem a provar un altre algorisme de stemming que funciona millor en castellà.
# S'anomena Snowball Stemmer i és una evolució del Porter, amb millors regles i adaptat a més idiomes.

from nltk.stem import SnowballStemmer
snowball = SnowballStemmer('spanish') 
snowball_stemmed_tokens = [snowball.stem(w) for w in filtered_tokens]
# Mostrem els 50 primers tokens després de fer stemming amb Snowball
for token in snowball_stemmed_tokens[:50]: 
    print(token)


ingeni
hidalg
don
quijot
manch
tas
juan
gall
andrad
escriban
cam
rey
señor
resid
consej
certif
doy
fe
vist
señor
del
libr
intitul
ingeni
hidalg
manch
compuest
miguel
cervant
saavedr
tas
cad
plieg
dich
libr
tres
maraved
medi
ochent
tres
plieg
dich
preci
mont
dich
libr
docient
novent
maraved
medi


In [48]:

# Lemmatització (amb POS per a precisió). 
# La lemmatització és més precisa que el stemming, ja que té en compte la part de la parla (POS) 
# de cada paraula. És a dir, si una paraula és un verb, un substantiu, un adjectiu, etc., 
# la lemmatització la reduirà a la forma canònica corresponent.

# Com hem vist que el POS tagging de NLTK no funciona bé en castellà
# anem a fer una funció que mapeja les etiquetes de POS d'anglès a les categories gramaticals 
# que utilitza WordNet per a la lemmatització. Això ens ajudarà a millorar la precisió de la lemmatització.
def get_wordnet_pos(tag):
    if tag.startswith('J'): return 'a' # Adjectiu
    elif tag.startswith('V'): return 'v' # Verb
    elif tag.startswith('N'): return 'n' # Substantiu
    elif tag.startswith('R'): return 'r' # Adverbi
    return 'n'

# Com la lemmatització tarda un poc en un text tan llarg, anem a aplicar l'etiquetatge gramatical
# i la lemmatització 
pos_tags = pos_tag(filtered_tokens) # Etiquetatge gramatical (POS tagging) per a cada paraula
lemmatizer = WordNetLemmatizer()
# anem a aplicar la lemmatització només a les primeres 50 paraules de filtered_tokens per veure
# com funciona i no esperar massa temps
lemmatized = [lemmatizer.lemmatize(w.lower(), get_wordnet_pos(tag)) 
              for w, tag in pos_tags if w.lower() in filtered_tokens[:50]]

# Mostrem les 50 primeres paraules després de fer lemmatització
for token in lemmatized[:50]: 
    print(token)



ingenioso
hidalgo
don
quijote
mancha
tasa
juan
gallo
andrada
escribano
cámara
rey
señor
residen
consejo
certifico
doy
fe
visto
señores
dél
libro
intitulado
ingenioso
hidalgo
mancha
compuesto
miguel
cervantes
saavedra
tasaron
cada
pliego
dicho
libro
tres
maravedís
medio
ochenta
tres
pliegos
dicho
precio
monta
dicho
libro
docientos
noventa
maravedís
medio


Podeu comprovar que la lemmatització en castellà utilitzant WordNet no funciona bé, ja que està dissenyada per l'anglès. En un altre exemple utilitzarem la llibreria SpaCy, que té un bon suport per al castellà i altres idiomes, i veurem com la lemmatització funciona molt millor.

### N-grames

Els n-grams són seqüències de n paraules consecutives en un text. Per exemple, els bigrams (n=2) de la frase "El gat menja el peix" serien: "El gat", "gat menja", "menja el", "el peix". Els n-grams són útils per a l'anàlisi de text, ja que ens permeten capturar les relacions entre paraules i identificar patrons en el llenguatge. Per exemple, els bigrams poden ajudar-nos a identificar expressions comunes o col·locacions, com ara "molt bé", "de fet", etc. Els n-grams també són utilitzats en models de llenguatge i en l'anàlisi de sentiments per a capturar el context i les relacions entre paraules.

In [36]:
# Anem a fer que ens mostre els n-grames

# Per exemple, anem a demanar els bigrames (seqüència de 2 paraules consecutives)
# Per fer-ho ràpid, anem a treure els bigrames 
bigrames = list(ngrams(filtered_tokens, 2))
print("\nBigrames:", bigrames[:10])  # Primers 10 bigrames
# ara que ens mostre trigrames (3 paraules consecutives)
trigrames = list(ngrams(filtered_tokens, 3))
print("\nTrigrames:", trigrames[:10])  # Primers 10 trigrames


Bigrames: [('ingenioso', 'hidalgo'), ('hidalgo', 'don'), ('don', 'quijote'), ('quijote', 'mancha'), ('mancha', 'tasa'), ('tasa', ','), (',', 'juan'), ('juan', 'gallo'), ('gallo', 'andrada'), ('andrada', ',')]

Trigrames: [('ingenioso', 'hidalgo', 'don'), ('hidalgo', 'don', 'quijote'), ('don', 'quijote', 'mancha'), ('quijote', 'mancha', 'tasa'), ('mancha', 'tasa', ','), ('tasa', ',', 'juan'), (',', 'juan', 'gallo'), ('juan', 'gallo', 'andrada'), ('gallo', 'andrada', ','), ('andrada', ',', 'escribano')]


### Anàlisi exploratòria de les dades

L'anàlisi exploratòria de les dades (EDA) és el procés d'examinar i comprendre les dades abans de realitzar qualsevol anàlisi o modelatge. En el context del processament del llenguatge natural, l'EDA pot incloure tasques com ara: 

- Comptar la freqüència de les paraules 
- Identificar les paraules més comunes 
- Analitzar la longitud dels textos 
- Visualitzar les relacions entre paraules 
- Identificar temes o categories en els textos

I moltes més utilitats que podem aplicar al nostre text.

In [ ]:
# Anàlisi exploratòria de les dades
from nltk.metrics import BigramAssocMeasures  
from nltk.collocations import BigramCollocationFinder
from nltk.probability import FreqDist

# En principi l'anàlisi s'hauria de fer sobre el text ja lemmatitzat, per no diferenciar paraules
# que signifiquen el mateix o que són variacions una de l'altra (per exemple, "gatos" i "gato", o
# "comer" i "comiendo").

# Com la lemmatització no ha funcionat bé en castellà utilitzant WordNet, anem a fer l'anàlisi 
# directament sobre els tokens filtrats sense stop words
# Anàlisi exploratòria 
fdist = FreqDist(filtered_tokens)
print("Top 5 tokens filtrats:", fdist.most_common(5))

# Situació dels bigrames més comuns
bigram_measures = BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(filtered_tokens)
finder.apply_freq_filter(1)  

print("\nTop 5 bigramas (raw_freq):")
# raw_freq és la mètrica més senzilla, que simplement compta la freqüència de cada bigrama.
print(finder.nbest(bigram_measures.raw_freq, 5))

print("\nTop 5 bigramas (likelihood_ratio):")
# likelihood_ratio és una mètrica més sofisticada que té en compte no només la freqüència del bigrama
# sinó també la freqüència de les paraules individuals que el formen, per avaluar si el bigrama
# és més significatiu en el seu context que per separat.
print(finder.nbest(bigram_measures.likelihood_ratio, 5))

# Longitud mitjana
# És una mesura que ens ajuda a entendre la complexitat del text.
# Un text amb una longitud mitjana de paraula més alta pot ser més complex i formal
# mentre que un text amb una longitud mitjana de paraula més baixa pot ser més senzill i informal.
print("\nLongitud mitjana:", round(sum(len(w) for w in filtered_tokens) / len(filtered_tokens), 2))


Top 5 tokens filtrats: [('don', 2647), ('quijote', 2175), ('sancho', 2148), ('si', 1966), ('dijo', 1808)]

Top 5 bigramas (raw_freq):
[('don', 'quijote'), ('dijo', 'don'), ('respondió', 'sancho'), ('sancho', 'panza'), ('respondió', 'don')]

Top 5 bigramas (likelihood_ratio):
[('don', 'quijote'), ('sancho', 'panza'), ('vuesa', 'merced'), ('caballeros', 'andantes'), ('respondió', 'sancho')]

Longitud mitjana: 6.18
